# News Article Sentiment Analysis Pipeline

This notebook processes news article titles and uses the FinBERT model to calculate sentiment scores. 

**Input**: CSV files with columns (headline, date) - format: `TICKER_news.csv`

**Output**: Daily sentiment + price data per ticker - format: `TICKER_news_sentiment_daily.csv`

**Model**: FinBERT (ProsusAI/finbert) - Financial domain-specific BERT

## 1. Imports and Setup

In [13]:
import os
import re
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
import yfinance as yf

In [ ]:
# Directorie definitions
BASE_DIR = Path.cwd().resolve()
if (BASE_DIR / "gdelt_news_data").exists():
    pass
elif (BASE_DIR.parent / "gdelt_news_data").exists():
    BASE_DIR = BASE_DIR.parent

INPUT_DIR = str(BASE_DIR / "gdelt_news_data")
OUTPUT_DIR = BASE_DIR / "processed_data" / "news_sentiment_daily"

print(f"BASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

BASE_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP
INPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\gdelt_news_data
OUTPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily


## 2. Load FinBERT Model

In [14]:
# Load FinBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

# Move model to GPU if available
device = torch.device("CUDA GPU" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Model loaded onto {device}")

Model loaded onto cuda


## 3. Preprocessing Function

Cleans news article titles using REG-EX by removing:
- URLs
- Ticker symbols (NASDAQ:XXX format)
- Exchange prefixes
- News source metadata
- Special characters

In [4]:
def preprocess_title(title):
    if pd.isna(title) or not isinstance(title, str):
        return None

    title = re.sub(r'\s+', ' ', title).strip()
    title = re.sub(r'http\S+|www\.\S+', '', title)
    title = re.sub(r'\(\s*(NASDAQ|NYSE|AMEX)\s*:\s*\w+\s*\)', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*[-|]\s*[A-Z]{1,5}\s*$', '', title)
    title = re.sub(r'^\s*(NASDAQ|NYSE|AMEX)\s*:\s*', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\|\s*[A-Z][a-z]+\s*$', '', title)
    title = re.sub(r'\s*-\s*[A-Z][a-z]+\s+[A-Z][a-z]+\s*$', '', title)
    title = re.sub(r'[^\w\s\.,!?\'\"\-]', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()

    if len(title) < 15:
        return None

    return title

## 4. Sentiment Scoring

Uses FinBERT to calculate sentiment scores from -1 (negative) to +1 (positive)

In [5]:
def get_sentiment_score(texts, batch_size=16):
    if not texts:
        return []

    scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.softmax(outputs.logits, dim=1)

        # FinBERT outputs: [negative, neutral, positive]
        # Score = P(positive) - P(negative)
        batch_scores = predictions[:, 2] - predictions[:, 0]
        scores.extend(batch_scores.cpu().numpy().tolist())

    return scores

## 5. File Processing Function

In [6]:
def process_file(input_path, ticker):
    print(f"\nProcessing: {os.path.basename(input_path)} (Ticker: {ticker})")

    df = pd.read_csv(input_path)

    if 'headline' not in df.columns or 'date' not in df.columns:
        print(f"Skipping: Missing 'headline' or 'date' column")
        return

    print("Preprocessing news article titles...")
    df['clean_headline'] = df['headline'].apply(preprocess_title)

    valid_df = df.dropna(subset=['clean_headline']).copy()

    if len(valid_df) == 0:
        print("No valid headlines found after preprocessing")
        return

    print(f"Processing {len(valid_df)} valid news headlines")

    headlines = valid_df['clean_headline'].tolist()
    scores = get_sentiment_score(headlines)
    valid_df['sentiment_score'] = scores

    valid_df['datetime'] = pd.to_datetime(valid_df['date'])
    valid_df['sentiment_score'] = valid_df['sentiment_score'].round(4)
    result = valid_df[['datetime', 'sentiment_score']].copy()
    result['ticker'] = ticker
    result = result.sort_values('datetime')
    return result

## 6. Helper Functions

In [7]:
def extract_ticker(filename):
    base = filename.replace('.csv', '')

    if '_news' in base:
        return base.split('_news')[0]
    elif '_' in base:
        parts = base.split('_')
        for part in parts:
            if part.isupper() and 1 <= len(part) <= 5:
                return part

    return None


def fetch_price_data(ticker, start_date, end_date):
    try:
        print(f"  Fetching price data for {ticker}...")
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index()

        if df.empty:
            print(f"  Warning: No price data available for {ticker}")
            return None

        if "Date" not in df.columns:
            print(f"  Warning: Date column not found for {ticker}")
            return None

        df["Date"] = pd.to_datetime(df["Date"])
        df["date"] = df["Date"].dt.strftime("%Y-%m-%d")

        price_df = df[["date"]].copy()

        if "Open" in df.columns:
            price_df["open_price"] = pd.to_numeric(df["Open"], errors="coerce")
        else:
            price_df["open_price"] = None

        if "Close" in df.columns:
            price_df["close_price"] = pd.to_numeric(df["Close"], errors="coerce")
        else:
            price_df["close_price"] = None

        return price_df

    except Exception as e:
        print(f"Error fetching price data for {ticker}: {e}")
        return None

## 7. Main Processing Pipeline

Steps:
1. Load news CSV files
2. Extract ticker symbols
3. Preprocess titles and calculate sentiment
4. Aggregate to daily average
5. Fetch price data
6. Merge and save

In [12]:
csv_files = [
    filename 
    for filename in os.listdir(INPUT_DIR) 
    if filename.endswith('.csv')
]

if not csv_files:
    print(f"No CSV files found in {INPUT_DIR}")
else:
    print(f"Found {len(csv_files)} news CSV files to process")

Found 20 news CSV files to process


In [ ]:
# Process each news file
results = []
for filename in tqdm(csv_files, desc="Processing news files"):
    input_path = os.path.join(INPUT_DIR, filename)

    ticker = extract_ticker(filename)
    if not ticker:
        print(f"\nSkipping {filename}: Could not extract the ticker symbol")
        continue

    try:
        result = process_file(input_path, ticker)
        if result is not None and not result.empty:
            results.append(result)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

if not results:
    print("\nNo sentiment records generated from news articles.")

Processing news files:   0%|          | 0/20 [00:00<?, ?it/s]


Processing: AAPL_news.csv (Ticker: AAPL)
Preprocessing news article titles...
Processing 4365 valid news headlines


Processing news files:   5%|▌         | 1/20 [00:02<00:56,  2.97s/it]


Processing: AMZN_news.csv (Ticker: AMZN)
Preprocessing news article titles...
Processing 2272 valid news headlines


Processing news files:  10%|█         | 2/20 [00:04<00:34,  1.94s/it]

...

Processing news files:  90%|█████████ | 18/20 [00:24<00:01,  1.04it/s]


Processing: WMT_news.csv (Ticker: WMT)
Preprocessing news article titles...
Processing 1039 valid news headlines


Processing news files:  95%|█████████▌| 19/20 [00:25<00:00,  1.19it/s]


Processing: XOM_news.csv (Ticker: XOM)
Preprocessing news article titles...
Processing 1187 valid news headlines


Processing news files: 100%|██████████| 20/20 [00:26<00:00,  1.30s/it]


In [15]:
# Combine and average daily sentiment
if results:
    combined = pd.concat(results, ignore_index=True)
    combined = combined.dropna(subset=["datetime", "ticker", "sentiment_score"])
    combined = combined.sort_values(["datetime", "ticker"])

    combined["date"] = pd.to_datetime(combined["datetime"]).dt.strftime("%Y-%m-%d")
    total_daily_sentiment = (
        combined.groupby(["date", "ticker"], as_index=False)["sentiment_score"]
        .mean()
        .rename(columns={"sentiment_score": "avg_sentiment"})
        .sort_values(["ticker", "date"])
    )

    print(f"\n Total Daily sentiment records: {len(total_daily_sentiment)}")
    display(total_daily_sentiment.head())


 Total Daily sentiment records: 11388


,date,ticker,avg_sentiment
0,2023-10-10,AAPL,0.626000
17,2023-10-11,AAPL,0.598583
34,2023-10-12,AAPL,0.231337
54,2023-10-13,AAPL,-0.278825
69,2023-10-14,AAPL,-0.391150


## 8. Fetch Price Data and Merge

In [16]:
if results:
    min_date = daily["date"].min()
    max_date = daily["date"].max()

    print(f"\nFetching price data for date range: {min_date} to {max_date}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tickers = sorted(daily["ticker"].unique().tolist())

    for ticker in tickers:
        print(f"\nProcessing ticker: {ticker}")
        ticker_sentiment = daily[daily["ticker"] == ticker][["date", "avg_sentiment"]].copy()

        price_data = fetch_price_data(ticker, min_date, max_date)

        if price_data is not None and not price_data.empty:
            ticker_data = ticker_sentiment.merge(price_data, on="date", how="left")
            print(f"Merged {len(ticker_data)} records with price data")
        else:
            ticker_data = ticker_sentiment.copy()
            ticker_data["open_price"] = None
            ticker_data["close_price"] = None
            print("No price data available, saving sentiment only")

        out_path = OUTPUT_DIR / f"{ticker}_news_sentiment_daily.csv"
        ticker_data.to_csv(out_path, index=False)
        print(f"Saved to: {out_path}")

    print(f"\nCompleted processing for {len(tickers)} tickers")
    print(f"Output directory: {OUTPUT_DIR}")


Fetching price data for date range: 2023-10-10 to 2025-10-10

Processing ticker: AAPL
  Fetching price data for AAPL...
Merged 701 records with price data
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\AAPL_news_sentiment_daily.csv

Processing ticker: AMZN
  Fetching price data for AMZN...
Merged 639 records with price data
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\AMZN_news_sentiment_daily.csv

Processing ticker: AVGO
  Fetching price data for AVGO...
Merged 652 records with price data
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\AVGO_news_sentiment_daily.csv

Processing ticker: BRK-B
  Fetching price data for BRK-B...
Merged 690 records with price data
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\BRK-B_news_sentiment_daily.csv

Processing ticker: GOOGL
 